# Datová manipulace v pandas — Titanic

V tomhle notebooku si projdeme tři skupiny užitečných funkcí na klasickém datasetu *Titanic*:

1. **Binning** — `pd.cut` a `pd.qcut` (jak rozdělit spojitou proměnnou do kategorií)
2. **Kontingenční tabulky** — `pd.crosstab` (jak křížit kategorie, počítat a agregovat)
3. **Transformace hodnot** — `apply`, `map` a `where` / `mask` (jak hromadně měnit hodnoty ve sloupcích)

Předpokládám, že už umíš načíst data, filtrovat řádky a vybírat sloupce. Teď jdeme o krok dál.

## 0. Načtení dat

Titanic je zabudovaný v seabornu, takže ho můžeme načíst jedním řádkem — žádný CSV soubor není potřeba.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = sns.load_dataset('titanic')
df.head()

In [ ]:
# Rychlý přehled, co v datech je
df.info()

Pár klíčových sloupců, se kterými budeme pracovat:

- `survived` — 0 / 1, jestli pasažér přežil
- `pclass` — třída (1 / 2 / 3)
- `sex` — male / female
- `age` — věk (obsahuje NaN!)
- `fare` — cena lístku
- `embarked` — přístav nalodění (C / Q / S)

---
## 1. Binning — z čísel uděláme kategorie

Často máme spojitou proměnnou (věk, cenu, příjem) a chceme ji rozdělit do **skupin**, abychom mohli porovnávat. Tomu se říká *binning* a v pandas na to máme dvě funkce:

| funkce | jak dělí | kdy použít |
|---|---|---|
| `pd.cut` | podle **hodnot** (hranice si určím já) | dávají smysl konkrétní prahy: dítě / dospělý, ceny do 50 / 50–100 / 100+ |
| `pd.qcut` | podle **kvantilů** (každý koš má stejně řádků) | chci, aby skupiny byly podobně velké (kvartily, decily, percentily) |

### 1.1 `pd.cut` — vlastní hranice

Rozdělíme věk do životních fází.

In [ ]:
# Hranice si volíme samy. labels musí mít o jeden prvek MÉNĚ než bins.
bins = [0, 12, 18, 35, 60, 100]
labels = ['dítě', 'teenager', 'mladý dospělý', 'střední věk', 'senior']

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

df[['age', 'age_group']].head(10)

In [ ]:
# Kolik lidí spadlo do které skupiny?
df['age_group'].value_counts().sort_index()

**Co si zapamatovat o `cut`:**

- Hranice jsou ve výchozím nastavení **zleva otevřené, zprava uzavřené**: `(0, 12]`, `(12, 18]`. Hodnota 12 padne do `dítě`, hodnota 13 do `teenager`. Chování přepneš parametrem `right=False`.
- `NaN` hodnoty (chybějící věk) zůstanou `NaN` i v nové kategorii — to je správně, nepřežene se to.
- Hodnoty mimo rozsah (např. záporné nebo > 100) se stanou `NaN`. Pokud chceš zachytit i krajní hodnoty, použij `include_lowest=True` nebo si rozšiř hranice (`bins=[-np.inf, 12, 18, ..., np.inf]`).

### 1.2 `pd.qcut` — stejně velké skupiny

Cenu lístku rozdělíme do **kvartilů** (4 stejně velké skupiny). Hodí se, když nevíš, kde rozumné hranice udělat, a chceš "levný / podprůměrný / nadprůměrný / drahý".

In [ ]:
df['fare_quartile'] = pd.qcut(df['fare'], q=4, labels=['Q1 nejlevnější', 'Q2', 'Q3', 'Q4 nejdražší'])

df[['fare', 'fare_quartile']].head()

In [ ]:
# Každá skupina má (zhruba) stejný počet lidí — v tom je rozdíl proti cut
df['fare_quartile'].value_counts().sort_index()

In [ ]:
# Když chceš vidět, jaké hranice qcut vybral, vrať retbins=True
_, hranice = pd.qcut(df['fare'], q=4, retbins=True)
print('Hranice kvartilů:', hranice)

**Cvičení 1:** Vytvoř sloupec `fare_decile` — rozděl `fare` do 10 stejně velkých skupin (decilů). Bez popisků (labels), ať vidíš výchozí intervaly.

In [ ]:
# Tvoje řešení:


---
## 2. Kontingenční tabulky — křížíme kategorie

Když chceš vidět, jak dvě (nebo víc) kategorií spolu souvisí — třeba *kolik žen z první třídy přežilo* — potřebuješ tabulku, kde jsou v řádcích jedna kategorie a ve sloupcích druhá. Na to v pandas máme **`pd.crosstab`**.

Crosstab umí dvě věci:
- ve **výchozím nastavení počítá** kombinace kategorií (kolik mužů v 1. třídě, kolik žen ve 2. třídě, …)
- s parametry `values=` a `aggfunc=` umí i **agregovat** číselný sloupec (průměrný věk, medián ceny, …)

### 2.1 Počty kombinací

Začneme jednoduše — kolik mužů a žen bylo v jednotlivých třídách?

In [ ]:
pd.crosstab(df['sex'], df['pclass'])

In [ ]:
# Přidáme součty řádků a sloupců
pd.crosstab(df['sex'], df['pclass'], margins=True, margins_name='Celkem')

In [ ]:
# Místo absolutních počtů procenta — normalize='index' = řádky se sečtou na 1
# (kolik % žen bylo v každé třídě, kolik % mužů...)
pd.crosstab(df['sex'], df['pclass'], normalize='index').round(3)

Možnosti pro `normalize`:
- `'index'` — řádky se sčítají na 1 (procenta v rámci řádku)
- `'columns'` — sloupce na 1
- `'all'` nebo `True` — celá tabulka na 1

Klasická otázka u Titanicu: **přežily častěji ženy?**

In [ ]:
pd.crosstab(df['sex'], df['survived'], normalize='index').round(3)

Z řádků: ~74 % žen přežilo, ~19 % mužů. Velký rozdíl.

Crosstab zvládne i **víc proměnných** v indexu nebo sloupcích — stačí poslat seznam:

In [ ]:
# Přežití podle pohlaví A třídy najednou
pd.crosstab([df['sex'], df['pclass']], df['survived'], normalize='index').round(3)

### 2.2 Agregace přes `values=` a `aggfunc=`

Default crosstab je počítání řádků. Když ale chceš **průměr, sumu, medián** z nějakého číselného sloupce, stačí přidat dva parametry: `values=` (co agreguju) a `aggfunc=` (jak agreguju).

Otázka: **jaký byl průměrný věk podle pohlaví a třídy?**

In [ ]:
pd.crosstab(
    df['sex'],          # do řádků
    df['pclass'],       # do sloupců
    values=df['age'],   # co agreguju
    aggfunc='mean'      # jak agreguju
).round(1)

Krásně vidíš, že v 1. třídě byli starší pasažéři než ve 3. třídě.

Místo `'mean'` můžeš dát `'median'`, `'sum'`, `'min'`, `'max'`, `'std'` nebo libovolnou funkci (např. `np.mean`).

In [ ]:
# Medián ceny lístku podle třídy a pohlaví
pd.crosstab(
    df['pclass'],
    df['sex'],
    values=df['fare'],
    aggfunc='median'
).round(2)

In [ ]:
# Míra přežití podle naší vlastní věkové skupiny a třídy
# (survived je 0/1, takže průměr = podíl přeživších)
pd.crosstab(
    df['age_group'],
    df['pclass'],
    values=df['survived'],
    aggfunc='mean'
).round(3)

**Cvičení 2:** Spočítej *medián* ceny lístku (`fare`) podle přístavu nalodění (`embarked`) v řádcích a třídy (`pclass`) ve sloupcích.

In [ ]:
# Tvoje řešení:


---
## 3. Transformace hodnot — `apply`, `map`, `where`

Tři způsoby, jak hromadně měnit hodnoty. Pletou se, ale logika je jasná:

| funkce | na čem funguje | k čemu |
|---|---|---|
| `map` | jen na **jednom sloupci** (Series) | jednoduché 1:1 přemapování (slovník nebo funkce) |
| `apply` | na sloupci **nebo celém DataFrame** | složitější funkce, vlastní logika, víc sloupců najednou |
| `where` / `mask` | na sloupci/DataFrame | podmíněné nahrazení ("tam, kde platí podmínka, nech / přepiš") |

### 3.1 `map` — slovníkové přemapování

Nejjednodušší případ: máš kódy a chceš je nahradit popisky.

In [ ]:
prevod = {'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'}
df['embarked_full'] = df['embarked'].map(prevod)

df[['embarked', 'embarked_full']].head()

**Pozor:** hodnoty, které ve slovníku nejsou, se stanou `NaN`. Pokud bys chtěla zachovat původní hodnotu pro neznámé klíče, použij `.map(prevod).fillna(df['embarked'])` nebo `.replace()`.

`map` přijímá i funkci:

In [ ]:
# Lambda funkce — krátký zápis funkce
df['fare_round'] = df['fare'].map(lambda x: round(x, 0))
df[['fare', 'fare_round']].head()

### 3.2 `apply` — vlastní funkce, klidně složitá

Když logika nevejde do slovníku, napíšeš si vlastní funkci.

In [ ]:
def cena_kategorie(fare):
    if pd.isna(fare):
        return 'neznámá'
    elif fare < 10:
        return 'levný'
    elif fare < 50:
        return 'střední'
    else:
        return 'drahý'

df['fare_kat'] = df['fare'].apply(cena_kategorie)
df['fare_kat'].value_counts()

*(Mimochodem — tohle bys mohla udělat i přes `pd.cut`. To je často lepší volba. `apply` si nechej na případy, kde `cut` nestačí.)*

**Apply přes víc sloupců najednou** — použij `axis=1` a uvnitř funkce sahej na řádek jako na slovník.

In [ ]:
# Velikost rodiny = sourozenci/partneři + rodiče/děti + samotný pasažér
df['family_size'] = df.apply(lambda row: row['sibsp'] + row['parch'] + 1, axis=1)

df[['sibsp', 'parch', 'family_size']].head()

**Pozn.:** Pro tenhle konkrétní příklad by stačilo `df['sibsp'] + df['parch'] + 1` (vektorizovaně, bez apply) — bylo by to **mnohem rychlejší**. `apply(axis=1)` je pomalý a sahá se po něm, až když to vektorizovaně nejde.

### 3.3 `where` a `mask` — podmíněné nahrazení

`where` a `mask` jsou dvojčata s opačnou logikou:

- **`where(podmínka, jinak)`** — *kde podmínka platí, nech hodnotu*; kde neplatí, nahraď `jinak`
- **`mask(podmínka, jinak)`** — *kde podmínka platí, nahraď*; kde neplatí, nech hodnotu

Příklad: chceme oříznout extrémně vysoké ceny lístků na 100 (tzv. *capping*).

In [ ]:
# where: tam, kde fare <= 100, nech původní hodnotu; jinde dej 100
df['fare_capped'] = df['fare'].where(df['fare'] <= 100, 100)

df[['fare', 'fare_capped']].sort_values('fare', ascending=False).head(10)

In [ ]:
# Stejný výsledek pomocí mask (opačně formulovaná podmínka)
df['fare_capped_v2'] = df['fare'].mask(df['fare'] > 100, 100)

(df['fare_capped'] == df['fare_capped_v2']).all()

In [ ]:
# Doplnění chybějícího věku průměrem — typický use case pro where/mask
prumerny_vek = df['age'].mean()
df['age_filled'] = df['age'].where(df['age'].notna(), prumerny_vek)

df['age_filled'].isna().sum()  # mělo by být 0

*(Tip: pro doplňování chybějících hodnot je idiomatičtější `fillna()`. `where`/`mask` má smysl pro obecnější podmínky než "je to NaN".)*

**Cvičení 3:** Vytvoř sloupec `is_child`, který bude `True` pro pasažéry mladší 18 let, jinak `False`. Vyzkoušej to dvěma způsoby — pomocí `apply` s vlastní funkcí, a pomocí prosté vektorizované podmínky `df['age'] < 18`. Porovnej výsledky.

In [ ]:
# Tvoje řešení:


---
## 4. Závěrečné cvičení — všechno dohromady

Spoj všechny tři techniky do jedné analýzy:

1. Pomocí `pd.cut` vytvoř sloupec `age_group` s kategoriemi `dítě (0-12)`, `mladý (13-30)`, `dospělý (31-60)`, `senior (61+)`.
2. Pomocí `map` přemapuj `sex` z `male`/`female` na `M`/`Ž`.
3. Pomocí `pd.crosstab` zobraz **míru přežití** podle `age_group` (řádky) a pohlaví (sloupce). *Tip: použij `values=` a `aggfunc='mean'`.*
4. Krátce napiš pod tabulku, co z výsledku vyčteš.

In [ ]:
# Tvoje řešení:


---
## Shrnutí — kdy co

**Binning** (spojité → kategorické):
- `pd.cut` — když mám smysluplné prahy (věk dětí, cenové kategorie)
- `pd.qcut` — když chci stejně velké skupiny (kvantily)

**Kontingenční tabulky:**
- `pd.crosstab` — počty kombinací kategorií (default), nebo procenta přes `normalize='index'/'columns'/'all'`.
- Stejný `crosstab` umí i agregovat číselný sloupec — stačí přidat `values=` a `aggfunc=` (`'mean'`, `'median'`, `'sum'`, …).

**Transformace:**
- `.map()` — jednoduché 1:1 přemapování přes slovník nebo funkci (jen Series)
- `.apply()` — vlastní funkce, klidně komplikovaná; `axis=1` pro práci s víc sloupci
- `.where()` / `.mask()` — podmíněné nahrazení (pamatuj: opačná logika)

**Zlaté pravidlo:** než sáhneš po `apply`, zkus, jestli to nejde vektorizovaně (`df['a'] + df['b']`, `df['x'].between(0, 10)`, …). Vektorizace je čitelnější a často desetkrát rychlejší.